<a href="https://www.kaggle.com/code/abhishekgodara/cafa-6-protein-prediction-score-0-377?scriptVersionId=292269648" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. OPTIMIZED OBO PARSER
# ==========================================
def parse_obo_parents(go_obo_path):
    """Parse OBO file with minimal memory footprint"""
    print(f"[1/5] Parsing OBO Ontology...")
    term_parents = {}
    roots = {'GO:0003674', 'GO:0008150', 'GO:0005575'}
    
    with open(go_obo_path, "r") as f:
        current_id = None
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current_id = None
            elif line.startswith("id: GO:"):
                current_id = line[4:].strip()
            elif line.startswith("is_a: GO:"):
                if current_id:
                    pid = line.split()[1]
                    if current_id not in term_parents:
                        term_parents[current_id] = set()
                    term_parents[current_id].add(pid)
            elif line.startswith("relationship: part_of GO:"):
                if current_id:
                    parts = line.split()
                    if len(parts) >= 3:
                        pid = parts[2]
                        if current_id not in term_parents:
                            term_parents[current_id] = set()
                        term_parents[current_id].add(pid)
    
    return term_parents, roots

def get_ancestors_map(term_parents):
    """Build ancestor map with iterative DFS to avoid recursion depth issues"""
    print("[2/5] Building Ancestor Map...")
    ancestors = {}
    
    # Precompute all terms including those that only appear as parents
    all_terms = set(term_parents.keys())
    for parents in term_parents.values():
        all_terms.update(parents)
    
    # Process in topological order (leaves to roots) for efficiency
    for term in tqdm(all_terms, desc="Building ancestors"):
        if term in ancestors:
            continue
            
        stack = [term]
        while stack:
            current = stack[-1]
            
            # If we already computed ancestors for this term, skip
            if current in ancestors:
                stack.pop()
                continue
            
            # Get parents
            parents = term_parents.get(current, set())
            
            # Check if all parents have their ancestors computed
            unprocessed_parents = [p for p in parents if p not in ancestors]
            
            if unprocessed_parents:
                # Need to process parents first
                stack.extend(unprocessed_parents)
            else:
                # All parents processed, compute ancestors for current term
                all_ancestors = set()
                for parent in parents:
                    all_ancestors.add(parent)
                    all_ancestors.update(ancestors.get(parent, set()))
                
                ancestors[current] = all_ancestors
                stack.pop()
    
    return ancestors

# ==========================================
# 2. OPTIMIZED PROCESSING LOGIC
# ==========================================
def propagate_scores(term_scores, ancestors_map, roots):
    """Propagate scores upwards in the ontology"""
    final_scores = term_scores.copy()
    
    # Sort terms by score descending for more efficient propagation
    sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)
    
    for term, score in sorted_terms:
        if score <= 0:
            continue
            
        # Get ancestors and propagate max score
        for anc in ancestors_map.get(term, set()):
            current = final_scores.get(anc, 0.0)
            if score > current:
                final_scores[anc] = score
    
    # Ensure roots are present with score 1.0
    if final_scores:
        for root in roots:
            final_scores[root] = 1.0
    
    return final_scores

def normalize_scores(scores_dict, roots):
    """Apply intelligent normalization to improve score distribution"""
    if not scores_dict:
        return scores_dict
    
    # Separate root and non-root scores
    root_scores = {r: 1.0 for r in roots if r in scores_dict}
    non_root_items = [(term, score) for term, score in scores_dict.items() 
                     if term not in roots]
    
    if not non_root_items:
        return {**root_scores}
    
    terms, scores = zip(*non_root_items)
    scores = np.array(scores, dtype=np.float32)
    
    # Strategy 1: Boost lower scores toward the maximum
    max_score = scores.max()
    if max_score > 0:
        # Scale so max becomes 0.95 (leaving room for roots at 1.0)
        target_max = 0.95
        if max_score < target_max:
            scale_factor = target_max / max_score
            scores = np.minimum(1.0, scores * scale_factor)
        
        # Strategy 2: Apply power transformation to sharpen distribution
        # This gives more weight to higher scores
        power = 1.5
        scores = np.power(scores, 1/power)  # Inverse power to boost mid-range scores
        
        # Strategy 3: Ensure minimum spread between top scores
        sorted_scores = np.sort(scores)[::-1]
        if len(sorted_scores) > 1:
            # Ensure top score is significantly higher than second
            if sorted_scores[0] - sorted_scores[1] < 0.1:
                sorted_scores[0] = min(1.0, sorted_scores[1] + 0.15)
        
        # Update scores
        scores = sorted_scores[np.argsort(np.argsort(scores)[::-1])]
    
    # Create final dictionary
    final_scores = {term: float(score) for term, score in zip(terms, scores)}
    final_scores.update(root_scores)
    
    return final_scores

def process_protein_group(pid, group, ancestors_map, roots):
    """Process predictions for a single protein"""
    if len(group) == 0:
        return []
    
    # Convert to dictionary
    term_scores = {}
    for _, row in group.iterrows():
        term_scores[row['go_term']] = float(row['score'])
    
    # Propagate scores
    propagated = propagate_scores(term_scores, ancestors_map, roots)
    
    # Normalize scores
    normalized = normalize_scores(propagated, roots)
    
    # Filter and prepare results
    results = []
    for term, score in normalized.items():
        if score >= 0.01:  # Slightly higher threshold to reduce noise
            results.append((pid, term, score))
    
    return results

# ==========================================
# 3. CHUNKED PROCESSING FOR LARGE FILES
# ==========================================
def process_large_file_chunked(input_path, ancestors_map, roots, chunksize=500000):
    """Process very large files in chunks to avoid memory issues"""
    print(f"[3/5] Loading and processing submission in chunks...")
    
    all_results = []
    processed_proteins = set()
    
    # First, let's check the file structure
    with open(input_path, 'r') as f:
        first_line = f.readline().strip()
        num_columns = len(first_line.split('\t'))
    
    print(f"Detected {num_columns} columns in input file")
    
    # Define column names based on actual structure
    if num_columns == 3:
        col_names = ['protein_id', 'go_term', 'score']
        usecols = [0, 1, 2]
    elif num_columns == 4:
        col_names = ['protein_id', 'go_term', 'score', 'key']
        usecols = [0, 1, 2]
    else:
        raise ValueError(f"Unexpected number of columns: {num_columns}")
    
    # Get total number of lines for progress bar
    print("Counting total lines...")
    total_lines = 0
    with open(input_path, 'r') as f:
        for _ in tqdm(f, desc="Counting lines"):
            total_lines += 1
    
    # Process in chunks
    chunk_iterator = pd.read_csv(input_path, sep='\t', header=None,
                                 names=col_names, usecols=usecols,
                                 chunksize=chunksize, dtype={'score': np.float32})
    
    chunk_count = 0
    for chunk in tqdm(chunk_iterator, total=total_lines/chunksize, desc="Processing chunks"):
        chunk_count += 1
        
        # Clean up any NaN values
        chunk = chunk.dropna(subset=['protein_id', 'go_term', 'score'])
        
        # Filter to only valid GO terms (starts with GO:)
        chunk = chunk[chunk['go_term'].str.startswith('GO:')]
        
        # Group by protein and process
        grouped = chunk.groupby('protein_id')
        
        for pid, group in grouped:
            # Skip if already processed (in case protein appears in multiple chunks)
            if pid in processed_proteins:
                continue
            
            results = process_protein_group(pid, group, ancestors_map, roots)
            all_results.extend(results)
            processed_proteins.add(pid)
        
        # Clear memory periodically
        if chunk_count % 10 == 0:
            gc.collect()
    
    # Convert to DataFrame
    final_df = pd.DataFrame(all_results, columns=['protein_id', 'go_term', 'score'])
    return final_df

# ==========================================
# 4. MAIN PIPELINE WITH OPTIMIZATIONS
# ==========================================
def main():
    # Paths
    OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
    SUBMISSION_INPUT = '/kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv'
    SUBMISSION_OUTPUT = 'submission_optimized.tsv'
    
    print("=" * 60)
    print("CAFA 6 Submission Optimizer")
    print("=" * 60)
    
    # 1. Load Ontology
    term_parents, roots = parse_obo_parents(OBO_PATH)
    print(f"Loaded {len(term_parents):,} terms from ontology")
    print(f"Root terms: {roots}")
    
    ancestors_map = get_ancestors_map(term_parents)
    print(f"Built ancestor map with {len(ancestors_map):,} terms")
    
    # 2. Check input file
    print(f"\n[3/5] Input file: {SUBMISSION_INPUT}")
    file_size = os.path.getsize(SUBMISSION_INPUT)
    print(f"File size: {file_size/1024/1024:.1f} MB")
    
    # Sample first few lines
    print("\nFirst 3 lines of input file:")
    with open(SUBMISSION_INPUT, 'r') as f:
        for i in range(3):
            line = f.readline().strip()
            print(f"  Line {i+1}: {line}")
    
    # 3. Process submission
    USE_CHUNKING = file_size > 50 * 1024 * 1024  # 50MB threshold
    
    if USE_CHUNKING:
        print(f"\nUsing chunked processing (file > 50MB)...")
        final_df = process_large_file_chunked(SUBMISSION_INPUT, ancestors_map, roots, chunksize=1000000)
    else:
        print(f"\nLoading entire file...")
        # Try to detect number of columns
        with open(SUBMISSION_INPUT, 'r') as f:
            first_line = f.readline().strip()
            num_cols = len(first_line.split('\t'))
        
        col_names = ['protein_id', 'go_term', 'score'] if num_cols == 3 else ['protein_id', 'go_term', 'score', 'key']
        usecols = [0, 1, 2]
        
        submission = pd.read_csv(SUBMISSION_INPUT, sep='\t', header=None,
                               names=col_names, usecols=usecols,
                               dtype={'score': np.float32})
        
        print(f"Loaded {len(submission):,} predictions")
        
        # Process
        print("Processing predictions...")
        all_results = []
        grouped = submission.groupby('protein_id')
        
        for pid, group in tqdm(grouped, total=len(grouped), desc="Processing proteins"):
            results = process_protein_group(pid, group, ancestors_map, roots)
            all_results.extend(results)
        
        final_df = pd.DataFrame(all_results, columns=['protein_id', 'go_term', 'score'])
    
    # 4. Post-processing and save
    print(f"\n[4/5] Post-processing {len(final_df):,} predictions...")
    
    # Remove duplicates (just in case)
    final_df = final_df.drop_duplicates(subset=['protein_id', 'go_term'])
    
    # Sort by protein_id and score (descending)
    final_df = final_df.sort_values(['protein_id', 'score'], 
                                   ascending=[True, False])
    
    # Optional: Limit predictions per protein to reduce file size
    MAX_PREDS_PER_PROTEIN = 2000
    if 'protein_id' in final_df.columns:
        final_df = final_df.groupby('protein_id').head(MAX_PREDS_PER_PROTEIN)
        print(f"Limited to {MAX_PREDS_PER_PROTEIN} predictions per protein")
    
    # 5. Save to file
    print(f"\n[5/5] Saving to {SUBMISSION_OUTPUT}...")
    final_df.to_csv(SUBMISSION_OUTPUT, sep='\t', index=False, header=False)
    
    # 6. Statistics
    print("\n" + "=" * 60)
    print("OPTIMIZATION COMPLETE")
    print("=" * 60)
    print(f"Output Statistics:")
    print(f"  • Total predictions: {len(final_df):,}")
    print(f"  • Unique proteins: {final_df['protein_id'].nunique():,}")
    print(f"  • Unique GO terms: {final_df['go_term'].nunique():,}")
    print(f"  • Score range: [{final_df['score'].min():.4f}, {final_df['score'].max():.4f}]")
    print(f"  • Mean score: {final_df['score'].mean():.4f}")
    print(f"  • Median score: {final_df['score'].median():.4f}")
    
    # Score distribution
    print(f"\nScore Distribution:")
    for threshold in [0.01, 0.1, 0.3, 0.5, 0.7, 0.9]:
        count = (final_df['score'] >= threshold).sum()
        percentage = (count / len(final_df)) * 100
        print(f"  • ≥{threshold:.2f}: {count:,} predictions ({percentage:.1f}%)")
    
    print(f"\nTop 10 predictions:")
    print(final_df.head(10).to_string(index=False))
    print(f"\n✅ Saved optimized submission to: {SUBMISSION_OUTPUT}")

if __name__ == "__main__":
    main()

CAFA 6 Submission Optimizer
[1/5] Parsing OBO Ontology...
Loaded 40,119 terms from ontology
Root terms: {'GO:0003674', 'GO:0008150', 'GO:0005575'}
[2/5] Building Ancestor Map...


Building ancestors:   0%|          | 0/40122 [00:00<?, ?it/s]

Built ancestor map with 40,122 terms

[3/5] Input file: /kaggle/input/cafa6-protein-function-enhanced-nb-v2/submission.tsv
File size: 1430.8 MB

First 3 lines of input file:
  Line 1: A0A009IHW8	GO:0003953	1.0052745413928434
  Line 2: A0A009IHW8	GO:0007165	1.0046859363465197
  Line 3: A0A009IHW8	GO:0016787	1.005138022349089

Using chunked processing (file > 50MB)...
[3/5] Loading and processing submission in chunks...
Detected 3 columns in input file
Counting total lines...


Counting lines: 0it [00:00, ?it/s]

Processing chunks:   0%|          | 0/40.796692 [00:00<?, ?it/s]